# Exploration des modèles concernant les données behaviour

Pour les données behaviour, de part leurs natures non annotées, nous nous penchons naturellement sur une approche non supervisée en premier lieu. 
Nous avons aussi remarqué que les données brutes étant très *"intéressantes"*, nous sommes d'abord parti sur une ACP (donc une approche de réduction de dimensions) pour amenuiser les quantités de données que le futur modèle de clustering devra utiliser. 
Enfin, nous avons standardisé les données avant l'ACP. 
Ces données viennent de *"simulations"* faites sur l'app par l'équipe de développement. Nous avons donc plusieurs jeux de données à fusionner.


## Plan du notebook :
- Packages
- Imports des données
- Fusion des données
- Standardisation des données et sauvegarde du modèle
- ACP : recherche du nombre adéquat de composantes en suivant plusieurs approches, sauvegarde du meilleur modèle et des données projetées
- Choix du modèle de clustering : Kmeans ou HDBSCAN, sauvegarde du meilleur modèle et des données projetées
- Exemple d'une prédiction en suivant ce pipeline

## Packages

In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score
import matplotlib.pyplot as plt

## Import des données

In [ ]:
# Import des données
FP = "../../data/features/behaviour_features.jsonl"
df = pd.read_json(FP, lines=True)
FP2 = "../../data/features/MNbehaviour_features.jsonl"
df2 = pd.read_json(FP2, lines=True)

## Fusion des données

In [ ]:
# Fusion des données
features = pd.DataFrame(df["features"].tolist())
features2 = pd.DataFrame(df2["features"].tolist())
features_concat = pd.concat([features, features2], ignore_index=True)
features_concat.head()

## Standardisation des données et sauvegarde du modèle

In [ ]:
# Standardisation des données
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features_concat)
features_scaled_df = pd.DataFrame(features_scaled, columns=features_concat.columns)
dump(scaler, "../../data/models/scaler_final.joblib")
features_scaled_df.head()

## ACP : recherche du nombre adéquat de composantes, sauvegarde du meilleur modèle et des données projetées

In [ ]:
# ACP : recherche du nombre adéquat de composantes
pca = PCA(random_state=42)
pca.fit(features_scaled_df)
cum_var = np.cumsum(pca.explained_variance_ratio_)
threshold = 0.95
n_components = np.argmax(cum_var >= threshold) + 1
print(f"Nombre optimal de composantes : {n_components}")
plt.figure(figsize=(8,5))
plt.plot(cum_var, marker='o')
plt.axhline(0.95, linestyle='--')
plt.axvline(n_components, linestyle='--')
plt.xlabel("Nombre de composantes")
plt.ylabel("Variance expliquée cumulée")
plt.title("Choix du nombre de composantes PCA")
plt.show()
# Sauvegarde du modèle PCA
pca_final = PCA(n_components=n_components, random_state=42)
pca_final.fit(features_scaled_df)
dump(pca_final, "../../data/models/pca_final.joblib")
features_pca = pca_final.transform(features_scaled_df)

In [ ]:
# Sauvegarde des données projetées (2 premières composantes + label)
features_scaled_fit = scaler.fit_transform(features_concat)
features_pca_fit = pca_final.fit_transform(features_scaled_fit)
first_two_components = features_pca_fit[:, :2]
labels_col = kmeans_final.labels_.reshape(-1, 1)
labeled_data = np.hstack((first_two_components, labels_col))
np.save("../../data/models/labeled_data.npy", labeled_data)

## Choix du modèle de clustering : KMeans ou HDBSCAN, sauvegarde du meilleur modèle et des données projetées

In [ ]:
# Choix du modèle de clustering : KMeans ou HDBSCAN
# KMeans
inertias = []
silhouette_scores = []
calinski_harabasz_scores = []
K = range(2, 15)
for k in K:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(features_pca)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(features_pca, kmeans.labels_))
    calinski_harabasz_scores.append(calinski_harabasz_score(features_pca, kmeans.labels_))
plt.plot(K, inertias, marker='o')
plt.xlabel("Nombre de clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.show()
plt.plot(K, silhouette_scores, marker='o')
plt.xlabel("Nombre de clusters (k)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score for KMeans")
plt.show()
plt.plot(K, calinski_harabasz_scores, marker='o')
plt.xlabel("Nombre de clusters (k)")
plt.ylabel("Calinski-Harabasz Score")
plt.title("Calinski-Harabasz Score for KMeans")
plt.show()
# Modèle final
kmeans_final = KMeans(n_clusters=6, random_state=42)
kmeans_final.fit(features_pca)
dump(kmeans_final, "../../data/models/kmeans_final.joblib")

## Exemple d'une prédiction en suivant ce pipeline

In [ ]:
# Exemple d'une prédiction en suivant ce pipeline
def predict_cluster(point):
    point_scaled = scaler.transform([point])
    point_pca = pca_final.transform(point_scaled)
    cluster = kmeans_final.predict(point_pca)
    return cluster[0], point_pca[0, 0], point_pca[0, 1]
# Exemple d'utilisation :
# predict_cluster(features_concat.iloc[0].values)

## Paramètres retenus pour les modèles
- ACP : 16 composantes principales
- KMeans : 6 clusters

Ces paramètres sont utilisés pour la sauvegarde des modèles finaux et la projection des données.